In [ ]:
import os

for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        print(f"{root}/", file)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
DATA_DIR  = '/kaggle/input/competitions/datathon-2026-round-1/'

traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
sale = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])
promotion = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date', 'end_date'])

promotion['start_date'] = pd.to_datetime(promotion['start_date'])
promotion['end_date'] = pd.to_datetime(promotion['end_date'])

test = pd.read_csv(DATA_DIR + 'sample_submission.csv', parse_dates=['Date'])

In [ ]:
all_dates = pd.date_range(start=sale['Date'].min(), end='2024-07-01')
promo_daily = pd.DataFrame({'Date': all_dates})

all_dates = pd.date_range(start=sale['Date'].min(), end='2024-07-01')
promo_daily = pd.DataFrame({'Date': all_dates})


def get_promo_info(d):
    active = promotion[(promotion['start_date'] <= d) & (promotion['end_date'] >= d)]
    return pd.Series([len(active), active['discount_value'].max() if not active.empty else 0])

promo_daily[['active_promo_count', 'max_discount']] = promo_daily['Date'].apply(get_promo_info)


#########################################
traffic_features = traffic.copy()
traffic_features.rename(columns={'date': 'Date'}, inplace=True)

available_traffic_cols = [c for c in ['sessions', 'unique_visitors', 'page_views'] if c in traffic_features.columns]

for col in available_traffic_cols:
    traffic_features[f'{col}_lag_1'] = traffic_features[col].shift(1)

cols_to_keep = ['Date'] + [c for c in traffic_features.columns if '_lag_1' in c]
traffic_final = traffic_features[cols_to_keep]

#####################################33
df_final = sale.merge(promo_daily, on='Date', how='left')
df_final = df_final.merge(traffic_final, on='Date', how='left')

df_final['month'] = df_final['Date'].dt.month
df_final['day_of_week'] = df_final['Date'].dt.dayofweek
df_final['is_weekend'] = (df_final['day_of_week'] >= 5).astype(int)
df_final['day_of_month'] = df_final['Date'].dt.day
df_final['rev_lag_365'] = df_final['Revenue'].shift(365)

df_final['is_payday'] = df_final['day_of_month'].isin([15, 30, 31]).astype(int)

for lag in [7, 14, 30]:
    df_final[f'rev_lag_{lag}'] = df_final['Revenue'].shift(lag)

df_final.fillna(0, inplace=True)
df_final['rev_lag_365'] = df_final['rev_lag_365'].fillna(df_final['Revenue'].mean())

print(df_final.columns.tolist())
print(df_final.tail())


In [ ]:
import optuna
import numpy as np

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression

###########33###############################3
ignored_cols = ['Date', 'Revenue', 'COGS']
features = [col for col in df_final.columns if col not in ignored_cols]
target = 'Revenue'

train_data = df_final[df_final['Date'] < '2022-01-01']
valid_data = df_final[df_final['Date'] >= '2022-01-01']

train_features_means = train_data.groupby('month')[features].mean()

X_train, y_train = train_data[features], train_data[target]
X_val, y_val = valid_data[features], valid_data[target]

print(f"num feature: {len(features)}")


#optina toi uu tham so mo hinh
def objective_lgb(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05),
        'num_leaves': trial.suggest_int('num_leaves', 20, 63), #them
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1
    }
    
    model_lgb = lgb.LGBMRegressor(**params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(100)]
    )
    return np.sqrt(mean_squared_error(y_val, model_lgb.predict(X_val)))

###################################
def objective_xgb(trial):
    params_xgb = {
        'n_estimators': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'tree_method': 'hist',
        'random_state': 42,
    }
    
    model_xgb = xgb.XGBRegressor(**params_xgb, early_stopping_rounds=100)
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    return np.sqrt(mean_squared_error(y_val, model_xgb.predict(X_val)))

#####################################
def objective_cat(trial):
    params_cat = {
        'iterations': 3000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': 42,
        'verbose': False,
        'early_stopping_rounds':100
    }
    model_cat = CatBoostRegressor(**params_cat)
    model_cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    return np.sqrt(mean_squared_error(y_val, model_cat.predict(X_val)))

##########################3#######3
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=30)


#ensemble động, tìm trọng số emsemble
#dùng mô hình với tham số tối ưu từ Optuna
best_lgb = lgb.LGBMRegressor(**study_lgb.best_params, n_estimators=3000, random_state=42)
best_xgb = xgb.XGBRegressor(**study_xgb.best_params, n_estimators=3000, random_state=42, early_stopping_rounds=100)
best_cat = CatBoostRegressor(**study_cat.best_params, iterations=3000, random_seed=42, early_stopping_rounds=100, verbose=False)

#fit lại mô hình
best_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100)], eval_metric='rmse')
best_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
best_cat.fit(X_train, y_train, eval_set=(X_val, y_val))

#Tạo tập dữ liệu cho Meta-Model (Ensemble động)
val_preds_metrix = np.column_stack([
    best_lgb.predict(X_val),
    best_xgb.predict(X_val),
    best_cat.predict(X_val)
])

# Meta-model học cách kết hợp dự đoán dựa trên kết quả thực tế y_val

# meta_model = Ridge(alpha=1.0)
meta_model = LinearRegression(positive=True)

meta_model.fit(val_preds_metrix, y_val)
final_pred = meta_model.predict(val_preds_metrix)

mae = mean_absolute_error(y_val, final_pred)
rmse = np.sqrt(mean_squared_error(y_val, final_pred))
r2 = r2_score(y_val, final_pred)

In [ ]:
print("-" * 30)
print(f"Dynamic Ensemble params: {meta_model.coef_}")
print(f"MAE Ensemble: {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R2 Score Ensemble: {r2:.4f}")
print("-" * 30)

In [ ]:
test['Date'] = pd.to_datetime(test['Date'])

test_final = test[['Date']].copy()
test_final['month'] = test_final['Date'].dt.month
test_final['day_of_week'] = test_final['Date'].dt.dayofweek
test_final['day_of_month'] = test_final['Date'].dt.day
test_final['is_weekend'] = (test_final['day_of_week'] >= 5).astype(int)
test_final['is_payday'] = test_final['day_of_month'].isin([15, 16, 30, 31]).astype(int)

if 'promo_daily' in locals() or 'promo_daily' in globals():
    test_final = test_final.merge(promo_daily, on='Date', how='left').fillna(0)


In [ ]:
from datetime import datetime

history = df_final[['Date', 'Revenue']].copy()
test_predictions = []

test_final = test_final.sort_values('Date')

for d in test_final['Date'].unique():
    d = pd.to_datetime(d)
    row = test_final[test_final['Date'] == d].copy()
    
    for lag in [7, 14, 30, 365]:
        past_date = d - pd.Timedelta(days=lag)
        past_val = history.loc[history['Date'] == past_date, 'Revenue']

        feature_means = train_data.groupby('month')[features].mean()
        if not past_val.empty:
            row[f'rev_lag_{lag}'] = past_val.values[0]
        else:
            row[f'rev_lag_{lag}'] = feature_means.loc[d.month, 'Revenue']
    
    for feat in features:
        if feat not in row.columns:
            avg_val = train_features_means.loc[d.month, feat]
            row[feat] = avg_val if pd.notna(avg_val) else 0

    X_test_step = row[features]
    
    p_lgb = best_lgb.predict(X_test_step)[0]
    p_xgb = best_xgb.predict(X_test_step)[0]
    p_cat = best_cat.predict(X_test_step)[0]
    
    test_meta_input = np.column_stack([p_lgb, p_xgb, p_cat])
    rev_pred = meta_model.predict(test_meta_input)[0]
    rev_pred = max(0, rev_pred)
    
    test_predictions.append(rev_pred)
    
    history = pd.concat([history, pd.DataFrame({'Date': [d], 'Revenue': [rev_pred]})], ignore_index=True)

##############################333

submission_df = test.copy()
submission_df['Revenue'] = test_predictions
ratio = df_final['COGS'].sum() / df_final['Revenue'].sum()
submission_df['COGS'] = submission_df['Revenue'] * ratio

current_time = datetime.now().strftime("%Y%m%d_%H%M")
file_name = f'submission_{current_time}.csv'

submission_df.to_csv(file_name, index=False)
print(f"load thành công file 'submission{current_time}.csv'")
